In [1]:
from pathlib import Path
import json
import configparser
import pprint
import time

# pip install slicerio
import slicerio.server

# packages for 3d
#   # for OCT 3D stuff and using in jupyter
#   - trame-jupyter-extension
#   - trame
#   - trame-vtk
#   - trame-vuetify
#   - ipywidgets
import numpy as np
import pandas as pd

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

from zipfile import ZipFile
import xml.etree.ElementTree as ET


In [2]:
# helper to deal with XML reading
class GeeElem(object):
    """Wrapper around an ElementTree element. a['foo'] gets the
       attribute foo, a.foo gets the first subelement foo."""
    def __init__(self, elem):
        self.etElem = elem

    def __getitem__(self, name):
        res = self._getattr(name)
        if res is None:
            raise(AttributeError, "No attribute named '%s'" % name)
        return res

    def __getattr__(self, name):
        res = self._getelem(name)
        if res is None:
            raise(IndexError, "No element named '%s'" % name)
        return res

    def _getelem(self, name):
        res = self.etElem.find(name)
        if res is None:
            return None
        return GeeElem(res)

    def _getattr(self, name):
        return self.etElem.get(name)

class GeeTree(object):
    "Wrapper around an ElementTree."
    def __init__(self, fname):
        self.doc = ET.parse(fname)

    def __getattr__(self, name):
        if self.doc.getroot().tag != name:
            raise(IndexError, "No element named '%s'" % name)
        return GeeElem(self.doc.getroot())

    def getroot(self):
        return self.doc.getroot()

In [3]:
#%% Load OCT study information
#folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')
# this is also folder name
study_name = '20250326_v2_test1'
folder_study = folder_octexport_root/study_name;
folder_study_processed = folder_study/'processed';
if not folder_study.exists:
    raise RuntimeError("Study not found")

study_info = dict(
    study_has_yat_log = len(list(folder_study.glob('YAT*.log')))>0,
    study_num_oct_files = len(list(folder_study.glob('{:s}*.oct'.format(study_name)))),
    study_num_vtk_files = len(list(folder_study.glob('{:s}*.vtk'.format(study_name)))),
    study_num_jpg_files = len(list(folder_study.glob('{:s}*.jpg'.format(study_name)))),
    study_has_an_ini_file = len(list(folder_study.glob('{:s}*.ini'.format(study_name))))>0,
    study_has_json_info_file = len(list(folder_study.glob('info.json'.format(study_name))))>0,
);
print('STUDY:',study_name)
study_info

STUDY: 20250326_v2_test1


{'study_has_yat_log': True,
 'study_num_oct_files': 40,
 'study_num_vtk_files': 0,
 'study_num_jpg_files': 0,
 'study_has_an_ini_file': False,
 'study_has_json_info_file': False}

In [4]:
#cam_sz_x,cam_sz_y = img.size
#scan_ini_CenterX = 0.403438812717808; # from 20250325_v2_test_DUMMY_0009_Mode3D.ini
#scan_ini_CenterY = 1.2;

# Read Directly From Thorlabs .oct File

In [4]:
from dataclasses import dataclass

@dataclass
class OctFile:
    """Class for returning read-in information from a 3D .OCT file"""
    pvvol: pv.ImageData
    sitkvol: sitk.Image
    scalars: np.ndarray
    image: Image
    cfg_oct_xml: object
    cfg_oct_probe : dict


In [5]:
def read_oct(file_oct,make_pv_volume=False,make_sitk_volume=False):
    # per thorlabs documentation, .oct files are just .zip files
    zip_file_path = file_oct;

    with ZipFile(zip_file_path, 'r') as zf:
        # for zf_file in zf.filelist:
        #     print(zf_file)
        
        # open and read Probe.ini file
        with zf.open('data/Probe.ini') as file:
            content = file.read().decode('utf-8')
            configprobe = configparser.ConfigParser();
            configprobe.read_string('[probe]\n'+content)
            cfg_oct_probe = {k:dict(v) for k,v in configprobe.items()};
            cfg_oct_probe = cfg_oct_probe['probe'];

        # open and read Header.xml file
        with zf.open('Header.xml') as file:
            cfg_oct_xml = GeeTree(file);
        
        # open and read the videoimage image
        cam_img_attribs = [e.attrib for e in cfg_oct_xml.Ocity.DataFiles.etElem if e.text=='data\\VideoImage.data'][0];
        with zf.open('data/VideoImage.data','r') as file:
            videodata = np.frombuffer(file.read(),dtype=np.uint8)
        videodata = videodata.reshape((
            int(cam_img_attribs['SizeX']),
            int(cam_img_attribs['SizeZ']),
            int(cam_img_attribs['BytesPerPixel']),
        ))
        # re-arrange to RGBA (Thorlabs format is BGRA)
        videodata = videodata[:,:,[2,1,0,3]]


        # open and parse the data/Intensity.data into a PyVista / VTK Image Data volume
        vol_img_attribs = [e.attrib for e in cfg_oct_xml.Ocity.DataFiles.etElem if e.text=='data\\Intensity.data'][0];
        vol_dimensions = (
            int(cfg_oct_xml.Ocity.Image.SizePixel.etElem[0].text),
            int(cfg_oct_xml.Ocity.Image.SizePixel.etElem[1].text),
            int(cfg_oct_xml.Ocity.Image.SizePixel.etElem[2].text),
        );
        vol_spacing_mm = (
            float(cfg_oct_xml.Ocity.Image.PixelSpacing.etElem[0].text),
            float(cfg_oct_xml.Ocity.Image.PixelSpacing.etElem[1].text),
            float(cfg_oct_xml.Ocity.Image.PixelSpacing.etElem[2].text),
        );
        with zf.open('data/Intensity.data','r') as file:
            scalars = np.frombuffer(file.read(),dtype=np.float32)

    print('Image Dims:{:s} PixelSpacing[mm]:{:s}'.format(str(vol_dimensions),str(vol_spacing_mm)))

    # make pyvista vtk volume image
    if(make_pv_volume):
        pvvol = pv.ImageData(dimensions=vol_dimensions,spacing=vol_spacing_mm);
        pvvol['OCTintensity'] = scalars;
    else:
        pvvol = None;
    
    # make simpleitk volume image
    if(make_sitk_volume):
        sitkvol = sitk.GetImageFromArray( scalars.reshape(vol_dimensions,order='F').transpose((2,1,0)) , isVector=False); # sitk uses opposite indexing
        sitkvol.SetSpacing(vol_spacing_mm);
    else:
        sitkvol = None;

    # make a PILLOW image from camera data
    newimg = Image.fromarray(videodata,mode='RGBA');

    return OctFile(
        pvvol = pvvol,
        sitkvol = sitkvol,
        scalars = scalars,
        image = newimg,
        cfg_oct_xml = cfg_oct_xml,
        cfg_oct_probe = cfg_oct_probe,
    );

# Process all files

In [6]:
#octfiles = list(folder_study.glob('{:s}*.oct'.format(study_name))))
octdatalist = [];
if(study_info['study_num_oct_files']>=1):
    octfiles = list(folder_study.glob('{:s}*.oct'.format(study_name)))
    for octfile in octfiles:
        octdata = read_oct(octfile,make_sitk_volume=True);
        octdatalist.append(octdata);

Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(1024, 245, 329) PixelSpacing[mm]:(0.003475, 0.020014, 0.020048)
Image Dims:(

# V2 Heater Device Test Log

In [7]:
if(study_info['study_has_yat_log']):
    file_yat_v2_logfile = list(folder_study.glob('YAT*.log'))[0];
    dflog = pd.read_csv(file_yat_v2_logfile,header=None,parse_dates=[0]);
    dflog.columns = ['ts','cpumillis','AmpT','AmpSP','AmpPWM','ValveT','ValveSP','ValvePWM','Volt','unknown']

In [8]:
if(study_info['study_has_yat_log']):
    list_oct_timestamps = [pd.Timestamp.fromtimestamp(int(x.cfg_oct_xml.Ocity.Acquisition.Timestamp.etElem.text),tz='GMT').tz_convert(None) for x in octdatalist];
    list_oct_experimentnumber = [int(x.cfg_oct_xml.Ocity.MetaInfo.ExperimentNumber.etElem.text)-1 for x in octdatalist];
    #dflog
    #print(dflog[['ts','cpumillis']])
    print( list_oct_experimentnumber )
    dfoct = pd.DataFrame({'oct_ts':list_oct_timestamps,'oct_expnum':list_oct_experimentnumber})

    dflogmerge = pd.merge_asof(left=dflog,right=dfoct,left_on='ts',right_on='oct_ts',direction='backward')

    # some formatting
    dflogmerge['cpumillis'] = pd.to_numeric(dflogmerge['cpumillis'] , errors='coerce');
    dflogmerge = dflogmerge.dropna(subset=['cpumillis'])
    dflogmerge['cpumillis'] = dflogmerge['cpumillis'].astype(int)
    #dflogmerge.

    # write to processed folder
    folder_study_processed.mkdir(exist_ok=True);
    dflogmerge.to_csv(folder_study_processed/'V2devicelogprocessed.csv',index=False)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]


In [9]:
pd.to_numeric(dflogmerge['cpumillis'] , errors='coerce')

0         500
1        1500
2        2500
3        3500
4        4500
        ...  
410    410500
411    411500
412    412500
413    413500
414    414500
Name: cpumillis, Length: 415, dtype: int64

In [10]:
dflogmerge[['cpumillis','oct_expnum']]

,cpumillis,oct_expnum
0,500,1
1,1500,2
2,2500,2
3,3500,2
4,4500,2
...,...,...
410,410500,39
411,411500,39
412,412500,39
413,413500,39


# Save-out the ITK Volume Image

In [81]:
# play with a single simpleitk image
if False:
    simg = octdatalist[10].sitkvol;
    print(simg)
    # import matplotlib.pyplot as plt
    # slice = sitk.GetArrayViewFromImage(simg)[:,:,150]
    # plt.imshow(slice,cmap='gray')

    writer = sitk.ImageFileWriter()
    writer.SetFileName(folder_study_processed/'{:s}_TEST2.nrrd'.format(study_name));
    writer.Execute(simg);

In [ ]:
#simg1 = sitk.Image(allConcatenated.shape,sitk.sitkFloat32);
del allConcatenated
del simg

In [ ]:
if False:
    # Save Each As A .VTK File
    if(study_info['study_num_oct_files']>=1):
        folder_study_processed.mkdir(exist_ok=True);
        for count,octdata in enumerate(sorted(octdatalist, key= lambda x: int(x.cfg_oct_xml.Ocity.Acquisition.Timestamp.etElem.text))):
            octdata_study = octdata.cfg_oct_xml.Ocity.MetaInfo.Study.etElem.text;
            octdata_timestr = time.strftime('%Y%m%dT%H%M%S',time.gmtime(int(octdata.cfg_oct_xml.Ocity.Acquisition.Timestamp.etElem.text)));
            
            fname = '{:s}_{:02d}_{:s}'.format(octdata_study,count,octdata_timestr);
            print(fname);

            octdata.volume.save(folder_study_processed/(fname+'.vtk'));

In [20]:
# save simpleitk concatenating the volumes, which Slicer will interpret as a volume time-sequence
if True:
    #simgjoined = sitk.JoinSeries([x.sitkvol for x in octdatalist]);
    #simgjoined = sitk.VectorJoi([x.sitkvol for x in octdatalist]);
    composer = sitk.ComposeImageFilter();
    simgjoined = composer.Execute([x.sitkvol for x in octdatalist]);

In [22]:
print(simgjoined)

VectorImage (000001FB933ADFB0)
  RTTI typeinfo:   class itk::VectorImage<float,3>
  Reference Count: 1
  Modified Time: 3140
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 3128
  UpdateMTime: 3139
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [1024, 245, 329]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [1024, 245, 329]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [1024, 245, 329]
  Spacing: [0.003475, 0.020014, 0.020048]
  Origin: [0, 0, 0]
  Direction: 
1 0 0
0 1 0
0 0 1

  IndexToPointMatrix: 
0.003475 0 0
0 0.020014 0
0 0 0.020048

  PointToIndexMatrix: 
287.77 0 0
0 49.965 0
0 0 49.8803

  Inverse Direction: 
1 0 0
0 1 0
0 0 1

  VectorLength: 40
  PixelContainer: 
    ImportImageContainer (000001FB92C2E460)
      RTTI typeinfo:   class itk::Impo

In [23]:
writer = sitk.ImageFileWriter()
writer.SetFileName(folder_study_processed/'{:s}.seq.nrrd'.format(study_name));
writer.Execute(simgjoined);

In [11]:
dir(writer)

['Abort',
 'AddCommand',
 'DebugOff',
 'DebugOn',
 'Execute',
 'GetCompressionLevel',
 'GetCompressor',
 'GetDebug',
 'GetFileName',
 'GetGlobalDefaultCoordinateTolerance',
 'GetGlobalDefaultDebug',
 'GetGlobalDefaultDirectionTolerance',
 'GetGlobalDefaultNumberOfThreads',
 'GetGlobalDefaultThreader',
 'GetGlobalWarningDisplay',
 'GetImageIO',
 'GetKeepOriginalImageUID',
 'GetName',
 'GetNumberOfThreads',
 'GetNumberOfWorkUnits',
 'GetProgress',
 'GetRegisteredImageIOs',
 'GetUseCompression',
 'GlobalDefaultDebugOff',
 'GlobalDefaultDebugOn',
 'GlobalWarningDisplayOff',
 'GlobalWarningDisplayOn',
 'HasCommand',
 'KeepOriginalImageUIDOff',
 'KeepOriginalImageUIDOn',
 'RemoveAllCommands',
 'SetCompressionLevel',
 'SetCompressor',
 'SetDebug',
 'SetFileName',
 'SetGlobalDefaultCoordinateTolerance',
 'SetGlobalDefaultDebug',
 'SetGlobalDefaultDirectionTolerance',
 'SetGlobalDefaultNumberOfThreads',
 'SetGlobalDefaultThreader',
 'SetGlobalWarningDisplay',
 'SetImageIO',
 'SetKeepOriginalIma

In [11]:
time.gmtime(int(octdatalist[0].cfg_oct_xml.Ocity.Acquisition.Timestamp.etElem.text))

time.struct_time(tm_year=2025, tm_mon=3, tm_mday=26, tm_hour=11, tm_min=54, tm_sec=55, tm_wday=2, tm_yday=85, tm_isdst=0)

In [ ]:
octdata.cfg_oct_xml.Ocity.MetaInfo.Study.etElem

In [ ]:
octdata.volume

# Process OCT Camera's RGB Images

In [12]:
octdata.cfg_oct_probe['camerascalingx']

'48.33446705'

In [13]:
octdata.cfg_oct_probe['camerascalingy']

'48.66782401'

In [67]:
# test reads
# import nrrd
# header = nrrd.read_header((folder_study_processed/'RGBSequence.seq.mrb').as_posix())

In [72]:
import slicerio.server
class slicerio_server_simon_wrapper():
    # slicer's rest HTTP API
    # Note, exec command must be turned on
    # https://slicer.readthedocs.io/en/latest/user_guide/modules/webserver.html
    
    # slicerio API
    # https://github.com/lassoan/slicerio/blob/main/slicerio/server.py

    # slicer APIs
    # https://slicer.readthedocs.io/en/latest/developer_guide/mrml_overview.html#mrml-scene

    # script repository
    # https://slicer.readthedocs.io/en/latest/developer_guide/script_repository.html
    # https://github.com/Slicer/Slicer/blob/main/Docs/developer_guide/script_repository/gui.md
    @staticmethod
    def exec(commandstring):
        api_url = f"http://127.0.0.1:{slicerio.server.SERVER_PORT}/slicer/exec"
        print(api_url,commandstring);
        response = slicerio.server.requests.get(api_url,params=dict(source=commandstring));
        #api_url+='?{:s}'.format(slicerio.server.requests.utils.quote(commandstring));
        #response = slicerio.server.requests.get(api_url);
        #response = slicerio.server.requests.get(api_url,params=dict(source=commandstring));
        slicerio.server._report_error(response)
        return response;
    # @staticmethod
    # def exec(commandstring):
    #     api_url = f"http://127.0.0.1:{slicerio.server.SERVER_PORT}/slicer/exec"
    #     print(api_url,commandstring);
    #     response = slicerio.server.requests.get(api_url,params=dict(source=commandstring));
    #     #api_url+='?{:s}'.format(slicerio.server.requests.utils.quote(commandstring));
    #     #response = slicerio.server.requests.get(api_url);
    #     #response = slicerio.server.requests.get(api_url,params=dict(source=commandstring));
    #     slicerio.server._report_error(response)
    #     return response;
    @staticmethod
    def version():
        api_url = f"http://127.0.0.1:{slicerio.server.SERVER_PORT}/slicer/system/version";
        response = slicerio.server.requests.get(api_url);
        slicerio.server._report_error(response)
        return response.json();

In [73]:
slicer_running = slicerio.server.is_server_running();
print('Slicer running?',slicer_running)
if not slicer_running:
    print('Starting slicer...')
    slicerio.server.start_server();
    slicer_running = slicerio.server.is_server_running();
    print('Slicer running?',slicer_running)

Slicer running? True


In [113]:
r = slicerio_server_simon_wrapper.version()
print(r)

{'applicationName': 'Slicer', 'applicationDisplayName': 'Slicer', 'applicationVersion': '5.8.1', 'releaseType': 'Stable', 'repositoryUrl': 'https://github.com/Slicer/Slicer', 'repositoryBranch': 'Slicer', 'revision': '33241', 'majorVersion': 5, 'minorVersion': 8, 'arch': 'amd64', 'os': 'win', 'isCustomMainApplication': False, 'mainApplicationName': 'Slicer', 'mainApplicationRepositoryUrl': 'https://github.com/Slicer/Slicer', 'mainApplicationRepositoryRevision': '11eaf62', 'mainApplicationRevision': '33241', 'mainApplicationMajorVersion': 5, 'mainApplicationMinorVersion': 8, 'mainApplicationPatchVersion': 1}


In [114]:
import slicerio.server

if(study_info['study_num_oct_files']>=1 and study_info['study_has_yat_log']):
    # CREATE A TIME_MOSAIC OF RGB
    folder_study_processed.mkdir(exist_ok=True);
    tmparrays = [];
    for count,octdata in enumerate(sorted(octdatalist, key= lambda x: int(x.cfg_oct_xml.Ocity.Acquisition.Timestamp.etElem.text))):
        octdata_study = octdata.cfg_oct_xml.Ocity.MetaInfo.Study.etElem.text;
        octdata_timestr = time.strftime('%Y%m%dT%H%M%S',time.gmtime(int(octdata.cfg_oct_xml.Ocity.Acquisition.Timestamp.etElem.text)));
        
        #fname = '{:s}_{:02d}_{:s}'.format(octdata_study,count,octdata_timestr);
        fname = '{:s}_{:04d}'.format(octdata_study,count);
        print(fname,octdata_timestr);

        #octdata.volume.save(folder_study_processed/(fname+'.vtk'));
        #octdata.image.save(folder_study_processed/(fname+'.png'))


        simgcam = sitk.GetImageFromArray(np.array(octdata.image)[:,:,0:3],isVector=True);

        # camerascalingx and camerascalingy are defined as pixels/mm in the OCT probe .ini file
        simgcam.SetSpacing(( 1/float(octdata.cfg_oct_probe['camerascalingx']), 1/float(octdata.cfg_oct_probe['camerascalingy'])));

        # write nrrd image
        writer = sitk.ImageFileWriter()
        fname = folder_study_processed/'{:s}_videocamera_{:04d}.nrrd'.format(study_name,count)
        writer.SetFileName(fname);
        writer.Execute(simgcam);

        # load image in slicer
        slicerio.server.file_load(fname)
        
        # delete image - # this is kinda important because slicer will assume files with _0001 suffixes all need to be loaded as zslices!
        Path(fname).unlink();
        #break;
        

#         tmparrays.append( np.array(octdata.image) );
#     tmparray = np.stack(tmparrays,axis=-1);
#     # pretend that tmparray is a volume image with one dimension as size 1, for easy slicer sequences support
#     tmparray = tmparray[:,:,np.newaxis,0:3,:]
#     # make simpleitk 5d image
#     simgcam = sitk.GetImageFromArray(tmparray,isVector=True);
# print(simgcam)
# writer = sitk.ImageFileWriter()
# writer.SetFileName(folder_study_processed/'{:s}_videoimage_test.seq.nrrd'.format(study_name));
# writer.Execute(simgcam);

20250326_v2_test1_0000 20250326T115455
20250326_v2_test1_0001 20250326T115505
20250326_v2_test1_0002 20250326T115515
20250326_v2_test1_0003 20250326T115525
20250326_v2_test1_0004 20250326T115535
20250326_v2_test1_0005 20250326T115545
20250326_v2_test1_0006 20250326T115555
20250326_v2_test1_0007 20250326T115605
20250326_v2_test1_0008 20250326T115615
20250326_v2_test1_0009 20250326T115625
20250326_v2_test1_0010 20250326T115635
20250326_v2_test1_0011 20250326T115645
20250326_v2_test1_0012 20250326T115655
20250326_v2_test1_0013 20250326T115705
20250326_v2_test1_0014 20250326T115715
20250326_v2_test1_0015 20250326T115725
20250326_v2_test1_0016 20250326T115735
20250326_v2_test1_0017 20250326T115745
20250326_v2_test1_0018 20250326T115755
20250326_v2_test1_0019 20250326T115805
20250326_v2_test1_0020 20250326T115815
20250326_v2_test1_0021 20250326T115825
20250326_v2_test1_0022 20250326T115835
20250326_v2_test1_0023 20250326T115845
20250326_v2_test1_0024 20250326T115855
20250326_v2_test1_0025 20

In [115]:
#simgcam.SetSpacing((float(octdata.cfg_oct_probe['camerascalingx']), float(octdata.cfg_oct_probe['camerascalingy'])))
print(simgcam)

VectorImage (000001459B6B7860)
  RTTI typeinfo:   class itk::VectorImage<unsigned char,2>
  Reference Count: 1
  Modified Time: 61384
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 0
  UpdateMTime: 0
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 2
    Index: [0, 0]
    Size: [648, 484]
  BufferedRegion: 
    Dimension: 2
    Index: [0, 0]
    Size: [648, 484]
  RequestedRegion: 
    Dimension: 2
    Index: [0, 0]
    Size: [648, 484]
  Spacing: [0.0206892, 0.0205475]
  Origin: [0, 0]
  Direction: 
1 0
0 1

  IndexToPointMatrix: 
0.0206892 0
0 0.0205475

  PointToIndexMatrix: 
48.3345 0
0 48.6678

  Inverse Direction: 
1 0
0 1

  VectorLength: 3
  PixelContainer: 
    ImportImageContainer (000001459E5C8170)
      RTTI typeinfo:   class itk::ImportImageContainer<unsigned __int64,unsigned char>
      Reference Count: 1
      Modif

In [116]:
# slicer get list of volume nodes that were loaded
mrml_vector_volume_nodes = slicerio.server.node_ids(class_name="vtkMRMLVectorVolumeNode")
print(mrml_vector_volume_nodes)

['vtkMRMLVectorVolumeNode2', 'vtkMRMLVectorVolumeNode1', 'vtkMRMLVectorVolumeNode3', 'vtkMRMLVectorVolumeNode4', 'vtkMRMLVectorVolumeNode5', 'vtkMRMLVectorVolumeNode6', 'vtkMRMLVectorVolumeNode7', 'vtkMRMLVectorVolumeNode8', 'vtkMRMLVectorVolumeNode9', 'vtkMRMLVectorVolumeNode10', 'vtkMRMLVectorVolumeNode11', 'vtkMRMLVectorVolumeNode12', 'vtkMRMLVectorVolumeNode13', 'vtkMRMLVectorVolumeNode14', 'vtkMRMLVectorVolumeNode15', 'vtkMRMLVectorVolumeNode16', 'vtkMRMLVectorVolumeNode17', 'vtkMRMLVectorVolumeNode18', 'vtkMRMLVectorVolumeNode19', 'vtkMRMLVectorVolumeNode20', 'vtkMRMLVectorVolumeNode21', 'vtkMRMLVectorVolumeNode22', 'vtkMRMLVectorVolumeNode23', 'vtkMRMLVectorVolumeNode24', 'vtkMRMLVectorVolumeNode25', 'vtkMRMLVectorVolumeNode26', 'vtkMRMLVectorVolumeNode27', 'vtkMRMLVectorVolumeNode28', 'vtkMRMLVectorVolumeNode29', 'vtkMRMLVectorVolumeNode30', 'vtkMRMLVectorVolumeNode31', 'vtkMRMLVectorVolumeNode32', 'vtkMRMLVectorVolumeNode33', 'vtkMRMLVectorVolumeNode34', 'vtkMRMLVectorVolumeNo

In [117]:
# slicer make sequence node
# use Slicer exec rest endpoint to add sequence node
cmdstr = 'mergedSequenceNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLSequenceNode", "RGB_Camera_Sequence")'.format();
#print(cmdstr)
slicerio_server_simon_wrapper.exec(cmdstr);

http://127.0.0.1:2016/slicer/exec mergedSequenceNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLSequenceNode", "RGB_Camera_Sequence")


In [118]:
# slicer show the list of sequence nodes (should be only one)
mrml_sequence_nodes = slicerio.server.node_ids(class_name="vtkMRMLSequenceNode")
print(mrml_sequence_nodes);
assert(len(mrml_sequence_nodes)==1)

['vtkMRMLSequenceNode1']


In [119]:
# add frames to sequence
for count,mrml_vector_volume_node in enumerate(mrml_vector_volume_nodes):
        # use Slicer exec rest endpoint to add to sequence
        cmdstr = 'slicer.util.getNode("{:s}").SetDataNodeAtValue(slicer.util.getNode("{:s}"), "{:d}")'.format("vtkMRMLSequenceNode1",mrml_vector_volume_node,count);
        #print(cmdstr)
        slicerio_server_simon_wrapper.exec(cmdstr);

http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode2"), "0")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode1"), "1")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode3"), "2")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode4"), "3")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode5"), "4")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode6"), "5")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtVal

In [120]:
# set further sequence parameters
slicerio_server_simon_wrapper.exec('slicer.util.getNode("vtkMRMLSequenceNode1").SetIndexName("frame")')
slicerio_server_simon_wrapper.exec('slicer.util.getNode("vtkMRMLSequenceNode1").SetIndexUnit("")')

http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetIndexName("frame")
http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceNode1").SetIndexUnit("")


<Response [200]>

In [121]:
# remove volumes from slicer wworkspace
for mrml_vector_volume_node in mrml_vector_volume_nodes:
    slicerio.server.node_remove(id=mrml_vector_volume_node)

In [ ]:
# # slicer sequence browser creation
# #mergedSequenceBrowserNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLSequenceBrowserNode", "Merged")


# # use Slicer exec rest endpoint to save scene
# cmdstr = 'mergedSequenceBrowserNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLSequenceBrowserNode", "RGB_Camera_SequenceBrowser")';
# slicerio_server_simon_wrapper.exec(cmdstr);

# # slicer show the list of sequence nodes (should be only one)
# mrml_sequencebrowser_nodes = slicerio.server.node_ids(class_name="vtkMRMLSequenceBrowserNode")
# print(mrml_sequencebrowser_nodes);
# #assert(len(mrml_sequence_nodes)==1)

http://127.0.0.1:2016/slicer/exec mergedSequenceBrowserNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLSequenceBrowserNode", "RGB_Camera_SequenceBrowser")
['vtkMRMLSequenceBrowserNode1']


In [ ]:
# # Slicer Exec - add sequence to sequence browser
# cmdstr = 'mergedSequenceBrowserNode.AddSynchronizedSequenceNode(mergedSequenceNode)';
# slicerio_server_simon_wrapper.exec(cmdstr);

http://127.0.0.1:2016/slicer/exec mergedSequenceBrowserNode.AddSynchronizedSequenceNode(mergedSequenceNode)


In [ ]:
# # Slicer Exec - set active sequence browser node
# cmdstr = 'slicer.modules.sequences.toolBar().setActiveBrowserNode(mergedSequenceBrowserNode)';
# slicerio_server_simon_wrapper.exec(cmdstr);

http://127.0.0.1:2016/slicer/exec slicer.modules.sequences.toolBar().setActiveBrowserNode(mergedSequenceBrowserNode)


In [ ]:
# # Slicer Exec - Show proxy node in slice viewers
# cmdstr = 'mergedProxyNode = mergedSequenceBrowserNode.GetProxyNode(mergedSequenceNode)';
# slicerio_server_simon_wrapper.exec(cmdstr);

# cmdstr = 'slicer.util.setSliceViewerLayers(background=mergedProxyNode)';
# slicerio_server_simon_wrapper.exec(cmdstr);


http://127.0.0.1:2016/slicer/exec mergedProxyNode = mergedSequenceBrowserNode.GetProxyNode(mergedSequenceNode)
http://127.0.0.1:2016/slicer/exec slicer.util.setSliceViewerLayers(background=mergedProxyNode)


In [109]:
# save
#slicer.util.saveScene(r'D:\SGProjects\NAATOS\OCTlocal\20250326_v2_test1\processed\test.seq.mrb')
#slicer.util.saveScene((folder_study_processed/'RGB_Camera_Sequence.seq.mrb'.format()).as_posix())

In [110]:
# save RGBCameraSequence to a file
#slicerio.server.file_save(id='vtkMRMLSequenceNode1',file_path=(folder_study_processed/'RGB_Camera_Sequence.seq.mrb'.format()));

In [122]:
# slicer save scene
# use Slicer exec rest endpoint to save scene
cmdstr = 'slicer.util.saveScene("{:s}")'.format((folder_study_processed/'RGB_Camera_Sequence.seq.mrb'.format()).as_posix());
#print(cmdstr)
slicerio_server_simon_wrapper.exec(cmdstr);

http://127.0.0.1:2016/slicer/exec slicer.util.saveScene("D:/SGProjects/NAATOS/OCTlocal/20250326_v2_test1/processed/RGB_Camera_Sequence.seq.mrb")


In [112]:
# end / done

In [124]:
# load-in the merged volume-sequence we wrote earlier
folder_study_processed/'{:s}.seq.nrrd'.format(study_name)
slicerio.server.file_load(
    file_path=(folder_study_processed/'{:s}.seq.nrrd'.format(study_name)).as_posix(),
    file_type='SequenceFile'
);

In [140]:
# slicer show the list of sequence browser nodes
mrml_sequencebrowser_nodes = slicerio.server.node_ids(class_name="vtkMRMLSequenceBrowserNode")
print(mrml_sequencebrowser_nodes);
#assert(len(mrml_sequence_nodes)==1)
for nodeid in mrml_sequencebrowser_nodes:
    print(slicerio.server.node_properties(id=nodeid)[0])

# choose the last sequence browser, which is the one for the 3D OCT volume
mrml_sequencebrowser_nodeid = mrml_sequencebrowser_nodes[-1];


# slicer show the list of sequence nodes (should be only one)
mrml_sequence_nodes = slicerio.server.node_ids(class_name="vtkMRMLSequenceNode")
print(mrml_sequence_nodes);
for nodeid in mrml_sequence_nodes:
    props = slicerio.server.node_properties(id=nodeid)[0];
    print(props)
    if(props['Name']=='RGB_Camera_Sequence'):
        mrml_sequence_rgb_nodeid = nodeid;

['vtkMRMLSequenceBrowserNode1', 'vtkMRMLSequenceBrowserNode2']
{'ID': 'vtkMRMLSequenceBrowserNode1', 'ClassName': 'vtkMRMLSequenceBrowserNode', 'Name': 'SequenceBrowser', 'Debug': False, 'MTime': 27452821, 'HideFromEditors': False, 'Selectable': True, 'Selected': False, 'UndoEnabled': {'Playback active': False, 'Playback rate (fps)': 10, 'Playback item skipping enabled': True, 'Playback looped': True, 'Selected item number': -1, 'Recording active': False, 'Recording on master modified only': False, 'Recording sampling mode': 'limitedToPlaybackFrameRate', 'Index display mode': '[indexValue]', 'Index display format': '%.2f', 'Sequence nodes': ''}}
{'ID': 'vtkMRMLSequenceBrowserNode2', 'ClassName': 'vtkMRMLSequenceBrowserNode', 'Name': '20250326_v2_test1_1 browser', 'Debug': False, 'MTime': 27478780, 'HideFromEditors': False, 'Selectable': True, 'Selected': False, 'UndoEnabled': False, 'Node references': {'dataNodeRef0': 'vtkMRMLScalarVolumeNode1', 'sequenceNodeRef0': 'vtkMRMLSequenceNode

In [141]:
# Slicer Exec - add sequence for rgb camera to sequence browser for 3D OCT volume
cmdstr = 'slicer.util.getNode("{:s}").AddSynchronizedSequenceNode(slicer.util.getNode("{:s}"))'.format(mrml_sequencebrowser_nodeid,mrml_sequence_rgb_nodeid);
slicerio_server_simon_wrapper.exec(cmdstr);

http://127.0.0.1:2016/slicer/exec slicer.util.getNode("vtkMRMLSequenceBrowserNode2").AddSynchronizedSequenceNode(slicer.util.getNode("vtkMRMLSequenceNode1"))


In [ ]:
slicerio.server.node_properties

[{'ID': 'vtkMRMLSequenceNode1',
  'ClassName': 'vtkMRMLSequenceNode',
  'Name': 'RGBCameraSequence',
  'Debug': False,
  'MTime': 2912194,
  'HideFromEditors': False,
  'Selectable': True,
  'Selected': False,
  'UndoEnabled': False,
  'Attributes': {'DataNodeClassName': 'vtkMRMLVectorVolumeNode'},
  'Node references': {},
  'indexName': 'time',
  'indexUnit': 's',
  'indexType': 'numeric',
  'numericIndexValueTolerance': 0.001,
  'indexValues': '0 ... 39 (40 items)'}]

In [202]:
cmdstr = 'slicer.util.getNode("{:s}").SetDataNodeAtValue(slicer.util.getNode("{:s}"), "{:d}")'.format("vtkMRMLSequenceNode1",mrml_vector_volume_nodes[1],1);
print(cmdstr)
slicerio_server_simon_wrapper.exec(cmdstr)

slicer.util.getNode("vtkMRMLSequenceNode1").SetDataNodeAtValue(slicer.util.getNode("vtkMRMLVectorVolumeNode1"), "1")


<Response [200]>

In [ ]:
slicerio.

# Slicer Control

In [ ]:
import slicerio.server
pathf = (folder_study_processed/'{:s}.seq.nrrd'.format(study_name)).as_posix();
nodeid = slicerio.server.file_load(pathf)

['vtkMRMLVectorVolumeNode2']

In [32]:
slicerio.server.node_properties('20250326_v2_test1')

[{'ID': 'vtkMRMLVectorVolumeNode2',
  'ClassName': 'vtkMRMLVectorVolumeNode',
  'Name': '20250326_v2_test1',
  'Debug': False,
  'MTime': 1069430,
  'HideFromEditors': False,
  'Selectable': True,
  'Selected': False,
  'UndoEnabled': False,
  'Node references': {'display [displayNodeRef]': 'vtkMRMLVectorVolumeDisplayNode2 vtkMRMLColorLegendDisplayNode1',
   'storage [storageNodeRef]': 'vtkMRMLVolumeArchetypeStorageNode2'},
  'StorageNodeIDs[0]': 'vtkMRMLVolumeArchetypeStorageNode2',
  'DisplayNodeIDs[0]': 'vtkMRMLVectorVolumeDisplayNode2',
  'DisplayNodeIDs[1]': 'vtkMRMLColorLegendDisplayNode1',
  'Spacing': [0.003475, 0.020014, 0.020048],
  'Origin': [0, 0, 0],
  'VoxelVectorType': 'undefined',
  'IJKToRASDirections': '',
  'ImageData': {'Debug': False,
   'Modified Time': 564281,
   'Reference Count': 16,
   'Registered Events': {'Registered Observers': {'vtkObserver (000002B1FC7ABA80)': {'Event': 33,
      'EventName': 'ModifiedEvent',
      'Command': '000002B182FBCF60',
      'Pr